In [3]:
import pandas as pd

big = r"C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\Moca\2026\Slope_MOCA_CALCUL_2026_05_27_all_participants\OUTPUT.SLOPES\01.Data.SLOPES.xlsx"
glu = r"C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\Moca\2026\Slope_MOCA_CALCUL_2026_06-14_relative_Glu\OUTPUT.SLOPES\01.Data.SLOPES_relative.xlsx"
isaora = r"C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\Moca\Isaora_slopes\OUTPUT.SLOPES.2\OUTPUT.SLOPES.2\01.Data.SLOPES_Isaora.xlsx"

def load_with_pscid(path):
    df = pd.read_excel(path)
    id_candidates = [col for col in df.columns if col.lower() == "pscid"]
    if not id_candidates:
        raise KeyError(f"No participant id column found in {path}. Available columns: {list(df.columns)}")
    id_col = id_candidates[0]
    df = df.rename(columns={id_col: "pscid"})
    df["pscid"] = df["pscid"].astype(str).str.strip()
    return df

def prefix_non_key_columns(df, prefix):
    return df.rename(columns={col: f"{prefix}{col}" for col in df.columns if col != "pscid"})

df_big = load_with_pscid(big)
df_glu = prefix_non_key_columns(load_with_pscid(glu), "glu_")
df_isaora = prefix_non_key_columns(load_with_pscid(isaora), "isaora_")

merged = (
    df_big
    .merge(df_glu, on="pscid", how="left")
    .merge(df_isaora, on="pscid", how="left")
)

print(merged.shape)
merged.head()

(423, 13)


,pscid,Slope_score_MOCA,Intercept_score_MOCA,Slope_score_MOCA.scolarite,Intercept_score_MOCA.scolarite,glu_Slope_score_MOCA,glu_Intercept_score_MOCA,glu_Slope_score_MOCA.scolarite,glu_Intercept_score_MOCA.scolarite,isaora_Slope_Moca_raw,isaora_Intercept_Moca_raw,isaora_Slope_Moca_cor,isaora_Intercept_Moca_cor
0,7546989,-0.020883,27.978157,-0.020605,27.959845,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4509950,-0.016827,28.702734,-0.016737,28.789993,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4201521,-0.019122,23.560946,-0.018145,24.443844,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3319420,-0.025704,23.421186,-0.025991,23.450179,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4904940,-0.020883,24.315982,-0.020605,24.340219,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Compare MOCA slope scores only for participants present in all 3 files.
# For the Isaora file, this uses the raw MOCA slope by default.
# If you want the corrected version instead, replace isaora_Slope_Moca_raw with isaora_Slope_Moca_cor below.

compare = (
    df_big[["pscid", "Slope_score_MOCA"]]
    .merge(df_glu[["pscid", "glu_Slope_score_MOCA"]], on="pscid", how="inner")
    .merge(df_isaora[["pscid", "isaora_Slope_Moca_raw"]], on="pscid", how="inner")
)

compare = compare.rename(
    columns={
        "Slope_score_MOCA": "Slope_big",
        "glu_Slope_score_MOCA": "Slope_glu",
        "isaora_Slope_Moca_raw": "Slope_isaora",
    }
)

print(f"Overlapping participants in all 3 files: {len(compare)}")
print("\nPearson correlation matrix:")
print(compare[["Slope_big", "Slope_glu", "Slope_isaora"]].corr(method="pearson"))

print("\nMean absolute differences:")
print(
    pd.Series({
        "big_vs_glu": (compare["Slope_big"] - compare["Slope_glu"]).abs().mean(),
        "big_vs_isaora": (compare["Slope_big"] - compare["Slope_isaora"]).abs().mean(),
        "glu_vs_isaora": (compare["Slope_glu"] - compare["Slope_isaora"]).abs().mean(),
    })
)

compare.head()

KeyError: "['Slope_score_MOCA'] not in index"